In [ ]:

These notes are based on the Advanced RAG: How Corrective RAG (CRAG) Solves Traditional RAG Problems tutorial by CampusX.

1. What is Corrective RAG (CRAG)?
Corrective RAG (CRAG) is an advanced RAG architecture designed to solve the "blind trust" problem in traditional RAG systems.
In standard RAG, the system assumes that whatever the vector database retrieves is relevant.
If the retrieved documents are actually irrelevant, the LLM will likely generate a hallucinated or incorrect response 

In [ ]:
CRAG introduces a Self-Correction or Evaluation layer that checks the quality of retrieved documents before passing them to the generator.

In [ ]:

These notes are based on the Advanced RAG: How Corrective RAG (CRAG) Solves Traditional RAG Problems tutorial by CampusX.

1. What is Corrective RAG (CRAG)?
Corrective RAG (CRAG) is an advanced RAG architecture designed to solve the "blind trust" problem in traditional RAG systems.
In standard RAG, the system assumes that whatever the vector database retrieves is relevant. If the retrieved documents are actually irrelevant, the LLM will likely generate a hallucinated or incorrect response 


CRAG introduces a Self-Correction or Evaluation layer that checks the quality of retrieved documents before passing them to the generator.

2. The Core Problem: Traditional RAG Failure
In a traditional RAG pipeline:

User Query is converted into a vector.

Vector DB performs a similarity search.

LLM generates an answer based on the retrieved context.

The Flaw: If the Vector DB returns "garbage" (noise/irrelevant data), the LLM generates "garbage" 
 Traditional RAG has no mechanism to "say no" to poor quality context.

In [ ]:
The CRAG Workflow (The 3 Paths)
After retrieval, CRAG uses a Retrieval Evaluator (often a smaller, faster LLM or a specialized model)
to grade the relationship between the query and the retrieved documents. It categorizes the results into three paths

In [ ]:
Evaluation	Action Taken	Source Used
Correct	Knowledge Refinement: Filter and keep only the most relevant parts of the document.	Internal Knowledge (Vector DB)
Ambiguous	Hybrid Search: Use internal documents but also trigger a web search for more context.	Internal + External (Web)
Incorrect	Complete Replacement: Discard internal documents and perform a deep web search.

In [ ]:
Implementation Logic with LangGraph
To build CRAG, you use LangGraph to create a state machine with nodes representing these logical steps.

Node Structure:
Retrieve Node: Fetches documents from the Vector Store.

Grade Documents Node: An LLM determines if the documents are correct, incorrect, or ambiguous.

Web Search Node: Triggered only if the grade is incorrect or ambiguous.

Generate Node: Produces the final answer using the refined or searched context.

In [ ]:
from langgraph.graph import StateGraph, END

# 1. Define the State
class GraphState(TypedDict):
    query: str
    documents: List[str]
    generation: str
    search_needed: bool

# 2. Define Nodes
def retrieve(state):
    # Logic to fetch from Vector DB
    return {"documents": retrieved_docs}

def grade_documents(state):
    # Logic: LLM checks if docs match query
    # If bad: set search_needed = True
    return {"search_needed": True/False, "documents": filtered_docs}

def web_search(state):
    # Logic: Use Tavily/Google API to find new info
    return {"documents": web_docs}

def generate(state):
    # Logic: LLM generates final answer
    return {"generation": answer}

# 3. Build Graph
workflow = StateGraph(GraphState)
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade", grade_documents)
workflow.add_node("web_search", web_search)
workflow.add_node("generate", generate)

# Define Edges with Conditional Logic
workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "grade")
workflow.add_conditional_edges(
    "grade",
    decide_to_generate, # Function that checks search_needed
    {
        "search": "web_search",
        "generate": "generate"
    }
)
workflow.add_edge("web_search", "generate")
workflow.add_edge("generate", END)

In [ ]:
Key Takeaways
Robustness: CRAG is significantly more reliable for production apps where data in the Vector DB might be incomplete 


Knowledge Refinement: Instead of sending the whole document, CRAG identifies the specific relevant "knowledge strips," reducing noise and token costs.

Agentic Nature: It transforms a static pipeline into an agent that can reason about its own data quality.

In [ ]:
graph TD
    %% Entry Point
    Start((User Query)) --> Retrieval[Retriever Node: Fetch Docs from Vector DB]
    
    %% Evaluation Logic
    Retrieval --> Evaluator{Retrieval Evaluator: Grade Docs}
    
    %% Path 1: Correct
    Evaluator -- "Grade: CORRECT" --> Refine[Knowledge Refinement: Filter & Strip Noise]
    Refine --> GenCorrect[Generate Final Answer]
    
    %% Path 2: Ambiguous
    Evaluator -- "Grade: AMBIGUOUS" --> Hybrid[Hybrid Search: DB Docs + Web Search]
    Hybrid --> GenAmbiguous[Generate Final Answer]
    
    %% Path 3: Incorrect
    Evaluator -- "Grade: INCORRECT" --> WebSearch[Knowledge Searching: Trigger Web Search]
    WebSearch --> GenIncorrect[Generate Final Answer]
    
    %% Final Output
    GenCorrect --> End((Final Response))
    GenAmbiguous --> End
    GenIncorrect --> End

    %% Styling
    style Evaluator fill:#f9f,stroke:#333,stroke-width:2px
    style WebSearch fill:#ff9,stroke:#333
    style Refine fill:#9f9,stroke:#333